In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
# from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from openai import OpenAI

In [2]:
load_dotenv(override=True)

groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key= os.getenv("GROQ_API_KEY"))
groq_model = "openai/gpt-oss-120b"

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key = "ollama")
ollama_model = "llama3.1:8b"

In [3]:
db_name = "vector_db"

In [4]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 76 files in the knowledge base
Total characters in knowledge base: 304,434


In [5]:
# How many tokens in all the documents?

encoding = tiktoken.get_encoding("cl100k_base")
# encoding = tiktoken.encoding_for_model(groq_model)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {groq_model}: {token_count:,}")

Total tokens for openai/gpt-oss-120b: 63,721


In [6]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [7]:
documents[0]

Document(metadata={'source': 'knowledge-base/products/Rellm.md', 'doc_type': 'products'}, page_content="# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.\n\n## Features\n\n### AI-Driven Analytics\nRellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intellige

In [8]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[298]}")

Divided into 413 chunks
First chunk:

page_content='# HR Record

# Nina Patel

## Summary
- **Date of Birth:** July 25, 1991
- **Job Title:** Business Intelligence Analyst
- **Location:** Chicago, Illinois
- **Current Salary:** $82,000

## Insurellm Career Progression
- **February 2021 - Present:** Business Intelligence Analyst
  - Builds dashboards and reports using Tableau and Looker for executive team
  - Analyzes business metrics across all product lines
  - Partners with sales and marketing teams on data-driven insights

- **August 2019 - January 2021:** Junior BI Analyst
  - Created SQL queries and basic reports for business stakeholders
  - Supported senior analysts with data extraction and validation
  - Maintained existing dashboards and fixed data quality issues

- **May 2017 - July 2019:** Data Analyst at RetailMetrics Inc.
  - Analyzed retail sales data and customer behavior patterns
  - Built Excel-based reporting tools for operations team' metadata={'source': 'knowledge-b

In [9]:
print(chunks[299])

page_content='- **May 2017 - July 2019:** Data Analyst at RetailMetrics Inc.
  - Analyzed retail sales data and customer behavior patterns
  - Built Excel-based reporting tools for operations team

## Annual Performance History
- **2023:** Rating: 3.5/5
  *Meets expectations but has room for growth. Delivered all required reports but showed limited proactivity in identifying new insights.*

- **2022:** Rating: 3.2/5
  *Below expectations. Struggled with complex SQL queries and missed several deadlines. Enrolled in advanced analytics training.*

- **2021:** Rating: 3.8/5
  *Solid performance in first year as BI Analyst. Demonstrated good grasp of business metrics and stakeholder communication.*

- **2020:** Rating: 4.0/5
  *Strong performance as Junior Analyst. Showed initiative and eagerness to learn new tools.*

- **2019:** Rating: 3.9/5
  *Good onboarding year. Quick to adapt to Insurellm's data systems.*' metadata={'source': 'knowledge-base/employees/Nina Patel.md', 'doc_type': 'emp

## PINECONE 

In [31]:
from pinecone import Pinecone, ServerlessSpec

load_dotenv()


pc = Pinecone(api_key= os.getenv("PINECONE_API_KEY"))

In [40]:
index_name = "developer-quickstart-py"

if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map":{"text": "chunk_text"}
        }
    )

In [55]:



index = pc.Index(index_name)  

batch_size = 96

records = []

for i, chunk in enumerate(chunks):

    records.append({
        "_id": str(i),
        "chunk_text": chunk.page_content,
        "doc_type": chunk.metadata.get("doc_type", ""),
        "source": chunk.metadata.get("source", "")
    })

# Upload in batches
for i in range(0, len(records), batch_size):

    batch = records[i:i + batch_size]

    index.upsert_records(
        namespace="ns1",
        records=batch
    )

    print(f"Uploaded batch {i//batch_size + 1}")

Uploaded batch 1
Uploaded batch 2
Uploaded batch 3
Uploaded batch 4
Uploaded batch 5


In [56]:
query = "Who works in UX Developer?"

results = index.search_records(
    namespace="ns1",
    inputs={
        "text": query
    },
    top_k=2
)

print(results)

SearchRecordsResponse(result=SearchResult(hits=[Hit(id='336', score=0.30959874391555786, fields={'chunk_text': '# HR Record\n\n# Jessica Liu\n\n## Summary\n- **Date of Birth:** April 30, 1996\n- **Job Title:** Frontend Developer\n- **Location:** Remote (Based in Seattle, Washington)\n- **Current Salary:** $92,000\n\n## Insurellm Career Progression\n- **July 2022 - Present:** Frontend Developer\n  - Develops user interfaces for Rellm reinsurance platform using React\n  - Implements responsive designs and ensures cross-browser compatibility\n  - Collaborates with UX designers and backend engineers\n\n- **January 2020 - June 2022:** Junior Frontend Developer\n  - Built UI components for internal tools and customer-facing applications\n  - Fixed bugs and improved performance of existing web applications\n  - Participated in code reviews and learned best practices\n\n- **June 2018 - December 2019:** Web Developer Intern at StartupLabs\n  - Created landing pages and marketing websites\n  - L

## HUGGING FACE

In [10]:
# Pick an embedding model

# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
embeddings = HuggingFaceEmbeddings(model="BAAI/bge-m3")

#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Vectorstore created with 413 documents


In [11]:
results = vectorstore.similarity_search(
    "Who works in UX Developer?",
    k=2 # top 2
)
print(results)

[Document(id='47a50bf3-ae52-4406-b515-30e1e4f9e06d', metadata={'source': 'knowledge-base/employees/Michelle Rivera.md', 'doc_type': 'employees'}, page_content='- **May 2013 - July 2016:** UX Designer at CreativeAgency\n  - Created user interfaces for consumer and business applications\n  - Developed skills in user research and interaction design\n\n## Annual Performance History\n- **2023:** Rating: 4.8/5\n  *Outstanding performance. Led major Rellm redesign that improved user efficiency by 40%. Exceptional user research and design execution.*\n\n- **2022:** Rating: 4.5/5\n  *Exceeded expectations. Strong design leadership and excellent collaboration with engineering team.*\n\n- **2021:** Rating: 4.3/5\n  *Strong performance. Successfully balanced multiple design projects and maintained high quality standards.*\n\n- **2020:** Rating: 4.1/5\n  *Good performance adapting to remote design collaboration. Maintained design quality during challenging year.*\n\n- **2019:** Rating: 4.0/5\n  *So

In [12]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 413 vectors with 1,024 dimensions in the vector store


## VISUALIZE

In [13]:
collection

Collection(name=langchain)

In [14]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [15]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vector Store Visualization',
    xaxis_title='X',
    yaxis_title='Y',
    width=800,
    height=600
)

import plotly.io as pio
pio.renderers.default = "browser"

fig.show()

In [84]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)
import plotly.io as pio
pio.renderers.default = "browser"
fig.show()